In [103]:
# ---- importing functions ----
import sys
import numpy as np

# importing slow dynamics pipeline
pipeline_dir = 'C:/Users/ys2605/Desktop/stuff/slow_dynamics_analysis'    # edit this  
sys.path.append(pipeline_dir + '/functions')
import sd_utils as sd

In [104]:
# ---- Save figures as along the way ---- 
save_figs = True  # can turn on or off to save figures or not
fig_dir = 'C:/Users/ys2605/Desktop/stuff/papers/AC_paper_protocol/figures/python_int'    # edit this
seed = 0           # integer for reproducible results, or None for random

In [105]:
# ---- loading mismatch CaIm datasets ----
# for external datasets these steps need to be modified
data_dir = 'F:/AC_data/caiman_data_missmatch/'   # edit this 
 
# loading raw firing rates, trial types, and stimuli times from oddball dataset
# returns a list of dicts (one per dataset); main field 'firing_rates' is (neurons, time)
data_ob = sd.load_caim_data_mat(
    data_dir,                            # data directory
    ext_list=['mat'],                    # data extension
    tags=['ammn', '_processed_data'],    # data tags
    num_files=20,                        # limit number of files to load
    deconvolution='oasis',               # deconvolution oasis or smoothdfdt
    smooth_std_duration=0.1)             # in sec

frame_rate = 1000/np.mean(sd.get_values(data_ob, 'volume_period')) # in Hz; edit this for external datasets

Loaded 20 datasets from 2 mice: [np.str_('M1'), np.str_('M2')]


In [106]:
# ---- loading RNN neuronal activity during control inputs ----
# three types of RNNs trainings: oddball recognition, control freq recognition, and untrained
# all three were tested with control inputs and neuronal activity was extracted
# returns a list of dicts (one per network); firing_rates is (neurons, time) if flatten_runs else (runs, neurons, time)
data_dir_rnn = 'F:/RNN_stuff/RNN_data/test_data/'
fname_rnn = 'RNN_test_data_2024_5_24_9h_42m2'
data_rnn = sd.load_rnn_test(
    data_dir_rnn, 
    fname_rnn + '_cont_data.npy',
    fname_rnn + '_params.npy',
    max_net_load = 5,               # max networks per type (lower = less memory)
    flatten_runs = False,           # keep runs separate as 3D (runs, neurons, time), for per-run INT
    cut_zero_trials = True,         # drop prepended zero-padding trials
    num_initial_trials_skip = 20,   # skip first N trials
    limit_network_types=['ob trained', 'freq trained', 'untrained'],     # types to load; [] = all
    seed=seed)                      # reproducibility seed (None = random)

frame_rate_rnn = 1000/np.mean(sd.get_values(data_rnn, 'volume_period')) # in Hz; edit this for external datasets
training_type = np.array(sd.get_values(data_rnn, 'training'))

Loaded 15 RNN networks: {np.str_('freq trained'): 5, np.str_('ob trained'): 5, np.str_('untrained'): 5}


In [107]:
# ---- Compute intrinsic timescales for CaIm data ----
tau_ob_net_all = []
tau_ob_cell_all = []
for n_dset in range(len(data_ob)):
    print('CaIm dset %d' %(n_dset))

    tau_net, tau_cell = sd.get_network_intrinsic_timescales(
        data_ob[n_dset]['firing_rates'],
        frame_rate)
    tau_ob_net_all.append(tau_net)
    tau_ob_cell_all.append(tau_cell)

CaIm dset 0
CaIm dset 1
CaIm dset 2
CaIm dset 3
CaIm dset 4
CaIm dset 5
CaIm dset 6
CaIm dset 7
CaIm dset 8
CaIm dset 9
CaIm dset 10
CaIm dset 11
CaIm dset 12
CaIm dset 13
CaIm dset 14
CaIm dset 15
CaIm dset 16
CaIm dset 17
CaIm dset 18
CaIm dset 19


In [108]:
# ---- Compute intrinsic timescales for RNN data ----

tau_rnn_net_all = []
tau_rnn_cell_all = []
for n_rnn in range(len(data_rnn)):
    print('rnn %d' %(n_rnn))
    
    tau_net, tau_cell = sd.get_network_intrinsic_timescales(
        data_rnn[n_rnn]['firing_rates'],
        frame_rate_rnn)
    tau_rnn_net_all.append(tau_net)
    tau_rnn_cell_all.append(tau_cell)

rnn 0
rnn 1
rnn 2
rnn 3
rnn 4
rnn 5
rnn 6
rnn 7
rnn 8
rnn 9
rnn 10
rnn 11
rnn 12
rnn 13
rnn 14


In [ ]:
# ---- Plotting intrinsic timescales for CaIm and RNN data ----
import importlib
importlib.reload(sd)

# do_log=True -> log y-scale ; do_log=False -> linear y-scale
fig, ax, groups = sd.plot_fig_tau_networks_comb(
    tau_ob_net_all,
    tau_rnn_net_all,
    training_type,
    tau_ob_cell_all,
    tau_rnn_cell_all,
    do_log=True)

# statistical comparison: prints result + draws bracket on the panel (scale auto-detected)
# test: 'ttest' (Welch), 'mannwhitney', 'wilcoxon'/'ttest_rel' (paired); alternative: 'greater'/'less'/'two-sided'
sd.stat_compare(ax, groups, 'CaIm net', 'CaIm neuron',
                test='mannwhitney', alternative='two-sided')
sd.stat_compare(ax, groups, 'CaIm neuron', 'Ob neuron',
                test='mannwhitney', alternative='two-sided')
sd.stat_compare(ax, groups, 'CaIm neuron', 'Freq neuron',
                test='mannwhitney', alternative='two-sided')
sd.stat_compare(ax, groups, 'CaIm net', 'Ob net',
                test='mannwhitney', alternative='two-sided')
sd.stat_compare(ax, groups, 'CaIm net', 'Freq net',
                test='mannwhitney', alternative='two-sided')

if save_figs:
    sd.save_fig(fig, path=fig_dir, name_tag='')